# Mesh Straightening Comparison: Bishop Frame vs. Blender Armature Skinning

This notebook demonstrates and compares two methods for mesh straightening:
1. **Bishop Frame Parallel Transport** (`animgen.animation.straight`): A mathematically exact, volume-preserving coordinate projection mapping. It does not suffer from linear interpolation artifacts.
2. **Blender Armature (`bpy`) Skinning**: Deforms the mesh using Blender's Armature modifier (Linear Blend Skinning / LBS) posed along relative rotations. It demonstrates the classic joint collapse (candy-wrapper distortion) at extreme bends.

We generate a curved cylinder mesh, straighten it with both methods, compute geometric metrics, and display them.

In [1]:
# Consolidated Imports
import sys
sys.path.append("..")

import numpy as np
import torch
import trimesh
from shapely.geometry import Point

from animgen.animation.straight import straighten, build_bishop_frame
from animgen.core.spline import Spline
from animgen.utils.math import rotation_matrix_from_vectors


In [2]:
# Step 1: Create a highly curved spline and sweep a cylinder along it
control_points = [
    [0.0, 0.0, 0.0],
    [0.5, 0.2, 1.0],
    [1.0, 0.8, 2.0],
    [0.8, 1.5, 3.0],
    [0.0, 1.8, 4.0],
    [-0.8, 1.2, 5.0],
    [-1.0, 0.0, 6.0],
]
pts = [torch.tensor(pt, dtype=torch.float32) for pt in control_points]
spline = Spline(pts, alpha=0.5)
curve_tensors = spline.evaluate_curve(num_points_per_segment=15)
curve_pts = np.array([t.detach().cpu().numpy() for t in curve_tensors])

radius = 0.3
circle_poly = Point(0, 0).buffer(radius, quad_segs=8)
curved_mesh = trimesh.creation.sweep_polygon(circle_poly, curve_pts)
print(f"Generated curved tube mesh with {len(curved_mesh.vertices)} vertices and {len(curved_mesh.faces)} faces.")
print("Showcasing curved mesh:")
curved_mesh.show()


Generated curved tube mesh with 2720 vertices and 5436 faces.
Showcasing curved mesh:


In [3]:
# Step 2: Straighten using NumPy (Bishop Frame)
print("Straightening using NumPy (Bishop Frame)...")
straightened_numpy = straighten(curved_mesh, spine_points=spline, axis="z")
print("NumPy straightening completed successfully!")


Straightening using NumPy (Bishop Frame)...
NumPy straightening completed successfully!


In [4]:
# Step 3: Calculate metrics for NumPy Bishop Frame
dists_xy_numpy = np.linalg.norm(straightened_numpy.vertices[:, :2], axis=1)
print("NumPy Straightening Stats (Expected: perfect cylinder of radius 0.3):")
print(f"  Target radius:               {radius}")
print(f"  Max radial distance:         {np.max(dists_xy_numpy):.4f}")
print(f"  Mean radial distance:        {np.mean(dists_xy_numpy):.4f}")
print(f"  Min radial distance:         {np.min(dists_xy_numpy):.4f}")
print("Showcasing NumPy straightened mesh:")
straightened_numpy.show()


NumPy Straightening Stats (Expected: perfect cylinder of radius 0.3):
  Target radius:               0.3
  Max radial distance:         0.3000
  Mean radial distance:        0.3000
  Min radial distance:         0.2998
Showcasing NumPy straightened mesh:


In [5]:
# Step 4: Define Blender Armature (bpy) LBS deformation for comparison
def deform_mesh_to_spine_bpy(
    mesh: trimesh.Trimesh,
    source_spine: np.ndarray,
    target_spine: np.ndarray,
) -> trimesh.Trimesh:
    """
    Deform a mesh using Blender's Armature modifier (Linear Blend Skinning / LBS) via bpy.
    """
    import bpy
    import mathutils

    N_points = len(source_spine)
    if N_points < 2:
        raise ValueError("Spine must have at least 2 points to define segments.")

    # 1. Compute vertex weights using optimized numpy binning
    A_src = source_spine[:-1]
    D_src = source_spine[1:] - source_spine[:-1]
    L2_src = np.sum(D_src**2, axis=1)
    L2_src = np.maximum(L2_src, 1e-12)

    disp = mesh.vertices[:, None, :] - A_src[None, :, :]
    dot = np.sum(disp * D_src[None, :, :], axis=2)
    t_val = np.clip(dot / L2_src[None, :], 0.0, 1.0)
    proj = A_src[None, :, :] + t_val[:, :, None] * D_src[None, :, :]
    dist2 = np.sum((mesh.vertices[:, None, :] - proj) ** 2, axis=2)
    closest_seg = np.argmin(dist2, axis=1)
    t_star = t_val[np.arange(len(mesh.vertices)), closest_seg]

    bins = np.linspace(0.0, 1.0, 21)
    bin_indices = np.digitize(t_star, bins) - 1

    # 2. Setup temporary Blender Armature and Mesh Objects
    arm_data = bpy.data.armatures.new("TempDeformArm")
    arm_obj = bpy.data.objects.new("TempDeformArm", arm_data)
    bpy.context.collection.objects.link(arm_obj)

    mesh_data = bpy.data.meshes.new("TempDeformMesh")
    mesh_obj = bpy.data.objects.new("TempDeformMesh", mesh_data)
    bpy.context.collection.objects.link(mesh_obj)

    try:
        # Create Edit Bones matching the source spine
        bpy.context.view_layer.objects.active = arm_obj
        bpy.ops.object.mode_set(mode="EDIT")

        for i in range(N_points - 1):
            bone = arm_data.edit_bones.new(f"Bone_{i}")
            bone.head = source_spine[i].tolist()
            bone.tail = source_spine[i + 1].tolist()
            if i > 0:
                bone.parent = arm_data.edit_bones[f"Bone_{i - 1}"]
                bone.use_connect = True

        bpy.ops.object.mode_set(mode="OBJECT")

        # Load mesh vertices and faces
        mesh_data.from_pydata(mesh.vertices.tolist(), [], mesh.faces.tolist())
        mesh_data.update()

        # Create Vertex Groups and assign weights
        vgs = [
            mesh_obj.vertex_groups.new(name=f"Bone_{i}") for i in range(N_points - 1)
        ]
        for i in range(N_points - 1):
            mask_seg = closest_seg == i
            if not np.any(mask_seg):
                continue
            if i < N_points - 2:
                for b_idx in range(len(bins)):
                    mask_bin = (bin_indices == b_idx) & mask_seg
                    indices = np.where(mask_bin)[0]
                    if len(indices) == 0:
                        continue
                    w_next = float(bins[b_idx])
                    w_curr = 1.0 - w_next
                    if w_curr > 0.0:
                        vgs[i].add(indices.tolist(), w_curr, "REPLACE")
                    if w_next > 0.0:
                        vgs[i + 1].add(indices.tolist(), w_next, "REPLACE")
            else:
                indices = np.where(mask_seg)[0]
                vgs[i].add(indices.tolist(), 1.0, "REPLACE")

        # Add Armature Modifier
        mod = mesh_obj.modifiers.new(name="Arm", type="ARMATURE")
        mod.object = arm_obj
        mod.use_vertex_groups = True

        # Apply target matrices in Pose Mode using relative rotations
        bpy.context.view_layer.objects.active = arm_obj
        bpy.ops.object.mode_set(mode="POSE")

        for i in range(N_points - 1):
            pb = arm_obj.pose.bones[f"Bone_{i}"]

            T_src = source_spine[i + 1] - source_spine[i]
            T_tgt = target_spine[i + 1] - target_spine[i]
            R_diff = rotation_matrix_from_vectors(T_src, T_tgt)

            M_bind = np.array(pb.bone.matrix_local)
            R_bind = M_bind[:3, :3]
            R_pose = R_diff @ R_bind

            M = np.eye(4)
            M[:3, :3] = R_pose
            M[:3, 3] = target_spine[i]
            pb.matrix = mathutils.Matrix(M.tolist())

        bpy.ops.object.mode_set(mode="OBJECT")

        # Get deformed vertices from Evaluated Mesh
        dg = bpy.context.evaluated_depsgraph_get()
        eval_mesh_obj = mesh_obj.evaluated_get(dg)
        eval_mesh = eval_mesh_obj.to_mesh()

        new_vertices = np.array([v.co for v in eval_mesh.vertices])
        eval_mesh_obj.to_mesh_clear()

    finally:
        # Clean up temporary datablocks and objects
        if mesh_obj.name in bpy.data.objects:
            bpy.data.objects.remove(mesh_obj, do_unlink=True)
        if arm_obj.name in bpy.data.objects:
            bpy.data.objects.remove(arm_obj, do_unlink=True)
        if mesh_data.name in bpy.data.meshes:
            bpy.data.meshes.remove(mesh_data, do_unlink=True)
        if arm_data.name in bpy.data.armatures:
            bpy.data.armatures.remove(arm_data, do_unlink=True)

    straightened_mesh = mesh.copy()
    straightened_mesh.vertices = new_vertices
    return straightened_mesh

# Step 4b: Run Blender Armature (bpy) straightening
print("Straightening using Blender Armature (bpy)...")
_, _, _, _, s = build_bishop_frame(curve_pts)
target_spine = np.zeros_like(curve_pts)
target_spine[:, 2] = s
straightened_bpy = deform_mesh_to_spine_bpy(curved_mesh, curve_pts, target_spine)
print("bpy straightening completed successfully!")


Straightening using Blender Armature (bpy)...
bpy straightening completed successfully!


In [6]:
# Step 5: Calculate metrics for bpy backend
dists_xy_bpy = np.linalg.norm(straightened_bpy.vertices[:, :2], axis=1)
print("Blender bpy Straightening Stats (Expected: collapsed volume/radius at curved joints):")
print(f"  Target radius:               {radius}")
print(f"  Max radial distance:         {np.max(dists_xy_bpy):.4f}")
print(f"  Mean radial distance:        {np.mean(dists_xy_bpy):.4f}")
print(f"  Min radial distance:         {np.min(dists_xy_bpy):.4f} (collapses toward 0 due to LBS/DQS!)")
print("Showcasing bpy straightened mesh:")
straightened_bpy.show()


Blender bpy Straightening Stats (Expected: collapsed volume/radius at curved joints):
  Target radius:               0.3
  Max radial distance:         0.9717
  Mean radial distance:        0.4485
  Min radial distance:         0.0252 (collapses toward 0 due to LBS/DQS!)
Showcasing bpy straightened mesh:


In [7]:
# Step 6: Comparison Summary
print("Comparison Summary:")
print(f"  NumPy Mean Radius:           {np.mean(dists_xy_numpy):.4f} (Ideal: 0.3000)")
print(f"  Blender bpy Mean Radius:      {np.mean(dists_xy_bpy):.4f} (Ideal: 0.3000)")
print(f"\n  NumPy Min Radius:            {np.min(dists_xy_numpy):.4f} (Ideal: 0.3000)")
print(f"  Blender bpy Min Radius:       {np.min(dists_xy_bpy):.4f} (Ideal: 0.3000)")


Comparison Summary:
  NumPy Mean Radius:           0.3000 (Ideal: 0.3000)
  Blender bpy Mean Radius:      0.4485 (Ideal: 0.3000)

  NumPy Min Radius:            0.2998 (Ideal: 0.3000)
  Blender bpy Min Radius:       0.0252 (Ideal: 0.3000)
